# Kochi Transit Network — Urban Equity Dashboard v3
### Fixed: route lines + heatmap now render correctly

**What was broken in v2 and is now fixed:**

| Bug | Root Cause | Fix |
|---|---|---|
| Heatmap invisible | Built from 610 stop points only | Now uses route shape geometry (~20k pts) |
| Route lines wrong colour | Python f-string `rgb(80,{g},255)` passed literally into JS | Colours computed as pure hex in Python |
| leaflet-heat CDN fails | CDN returns 403 in some environments | Library embedded inline — zero dependencies |

**Run all cells: Runtime > Run all**

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 1 — Install packages
# ═══════════════════════════════════════════════════════════════
!pip install requests numpy pandas scipy --quiet
print("Done.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 2 — Download mdb-1835 (Kochi buses + ferries) from MobilityData
# ═══════════════════════════════════════════════════════════════
import os, zipfile, requests

GTFS_URL = ("https://files.mobilitydatabase.org/mdb-1835/"
            "mdb-1835-202402080147/mdb-1835-202402080147.zip")
GTFS_ZIP = "/content/kochi.zip"
GTFS_DIR = "/content/kochi_gtfs"
OUT_DIR  = "/content/output"
os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.exists(GTFS_ZIP):
    print("Downloading Kochi GTFS (~5 MB)...")
    r = requests.get(GTFS_URL, stream=True, timeout=120)
    r.raise_for_status()
    with open(GTFS_ZIP,"wb") as f:
        for chunk in r.iter_content(8192): f.write(chunk)
    print("  Downloaded.")

if not os.path.exists(GTFS_DIR):
    with zipfile.ZipFile(GTFS_ZIP) as zf: zf.extractall(GTFS_DIR)

files = os.listdir(GTFS_DIR)
print(f"\nExtracted {len(files)} files:")
for fn in sorted(files):
    sz = os.path.getsize(os.path.join(GTFS_DIR,fn))
    print(f"  {fn:30s}  {sz//1024:>6} KB")


In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 3 — Load GTFS tables and build route network
#  FIX: route colors now computed correctly as hex strings
#       (previous version had a Python f-string bug in JS output)
# ═══════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
from collections import defaultdict

def load(fn):
    fp = os.path.join(GTFS_DIR, fn)
    if not os.path.exists(fp): return pd.DataFrame()
    df = pd.read_csv(fp, dtype=str).fillna("")
    df.columns = df.columns.str.strip()
    for c in df.columns: df[c] = df[c].str.strip()
    return df

print("Loading GTFS files...")
stops_df      = load("stops.txt")
routes_df     = load("routes.txt")
trips_df      = load("trips.txt")
stop_times_df = load("stop_times.txt")
shapes_df     = load("shapes.txt")
frequencies_df= load("frequencies.txt")

for col in ["stop_lat","stop_lon"]:
    stops_df[col] = pd.to_numeric(stops_df[col], errors="coerce")
for col in ["shape_pt_lat","shape_pt_lon","shape_pt_sequence"]:
    shapes_df[col] = pd.to_numeric(shapes_df[col], errors="coerce")

stops_df = (stops_df.dropna(subset=["stop_lat","stop_lon"])
            .pipe(lambda d: d[~((d.stop_lat==0)&(d.stop_lon==0))])
            .set_index("stop_id"))

print(f"  stops: {len(stops_df):,}  |  shapes pts: {len(shapes_df):,}  "
      f"|  trips: {len(trips_df):,}  |  routes: {len(routes_df):,}")

# ── Trip frequency (headway -> trips/hr) ─────────────────────
trip_freq = {}
if not frequencies_df.empty and "headway_secs" in frequencies_df.columns:
    frequencies_df["headway_secs"] = pd.to_numeric(
        frequencies_df["headway_secs"], errors="coerce").fillna(1800)
    for _, row in frequencies_df.iterrows():
        tph = 3600 / max(float(row["headway_secs"]), 60)
        tid = row["trip_id"]
        trip_freq[tid] = trip_freq.get(tid, 0) + tph

# ── Mappings ─────────────────────────────────────────────────
trip_to_shape = dict(zip(trips_df["trip_id"], trips_df.get("shape_id", pd.Series())))
trip_to_route = dict(zip(trips_df["trip_id"], trips_df["route_id"]))

shape_trip_count  = defaultdict(int)
shape_freq_weight = defaultdict(float)
shape_to_route    = {}
for _, row in trips_df.iterrows():
    tid = row["trip_id"]
    sid = row.get("shape_id","")
    if not sid: continue
    shape_trip_count[sid]  += 1
    shape_freq_weight[sid] += trip_freq.get(tid, 1.0)
    if sid not in shape_to_route:
        shape_to_route[sid] = row["route_id"]

# ── Route metadata ────────────────────────────────────────────
route_color_map = {}
route_type_map  = {}
for _, row in routes_df.iterrows():
    rid = row["route_id"]
    rc  = row.get("route_color","").strip()
    rt  = row.get("route_type","3").strip()
    route_type_map[rid] = rt
    route_color_map[rid] = ("#" + rc) if (rc and len(rc)==6) else None

# ── Build shape polylines ─────────────────────────────────────
print("Building route polylines from shapes.txt...")
shape_lines = {}
for sid, grp in shapes_df.groupby("shape_id"):
    grp_s  = grp.sort_values("shape_pt_sequence").dropna(subset=["shape_pt_lat","shape_pt_lon"])
    coords = list(zip(grp_s["shape_pt_lat"], grp_s["shape_pt_lon"]))
    if len(coords) >= 2:
        shape_lines[sid] = coords

max_w = max((shape_freq_weight.get(s,1) for s in shape_lines), default=1)
print(f"  Polylines: {len(shape_lines):,}  |  Max weight: {max_w:.1f}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 4 — Build heatmap from route geometry + prepare route JSON
#
#  FIX 1: Heatmap built from shape points (not stops)
#  FIX 2: Route colors computed as pure hex strings in Python
#         so no Python f-string survives into JS template
#  FIX 3: leaflet-heat will be embedded inline (no CDN)
# ═══════════════════════════════════════════════════════════════
import json, colorsys

# ── A. HEATMAP: dense points from route geometry ──────────────
print("Building heatmap from route geometry...")
heatmap_pts = []
KOCHI_LAT = (9.5, 10.6)
KOCHI_LON = (75.9, 76.7)

for sid, coords in shape_lines.items():
    raw_w  = shape_freq_weight.get(sid, shape_trip_count.get(sid, 1))
    norm_w = min(1.0, raw_w / max_w)
    intens = max(0.12, norm_w)
    step   = 2 if intens > 0.4 else 4
    for lat, lon in coords[::step]:
        if not (KOCHI_LAT[0] < lat < KOCHI_LAT[1]): continue
        if not (KOCHI_LON[0] < lon < KOCHI_LON[1]): continue
        heatmap_pts.append([round(lat,5), round(lon,5), round(intens,3)])

print(f"  Heat points: {len(heatmap_pts):,}  (was 610 with stop-only)")

# ── B. ROUTE LINES: pure Python hex color, no f-string in JS ─
print("Building route line objects...")

def freq_to_hex(norm_w, route_type):
    """Returns a hex color string computed entirely in Python."""
    if route_type == "4":               # ferry
        return "#00d4ff"
    # Bus: interpolate navy -> electric-blue as frequency rises
    # low freq = dim desaturated blue, high freq = bright vivid blue
    r = int(30  + norm_w * 20)
    g = int(80  + norm_w * 100)
    b = int(160 + norm_w * 95)
    return "#{:02x}{:02x}{:02x}".format(r, g, b)

route_lines_js = []
for sid, coords in shape_lines.items():
    rid    = shape_to_route.get(sid,"")
    rtype  = route_type_map.get(rid,"3")
    rcolor = route_color_map.get(rid, None)
    raw_w  = shape_freq_weight.get(sid, shape_trip_count.get(sid, 1))
    norm_w = min(1.0, raw_w / max_w)

    # Compute color entirely in Python (no f-string leaking into JS)
    color  = rcolor if rcolor else freq_to_hex(norm_w, rtype)
    lwidth = round(0.5 + norm_w * 2.5, 2)     # 0.5–3.0 px
    opac   = round(0.20 + norm_w * 0.65, 2)   # 0.20–0.85

    step   = max(1, len(coords)//60)
    pts    = coords[::step]
    if len(pts) < 2: continue

    route_lines_js.append({
        "c": [[round(la,5), round(lo,5)] for la,lo in pts],
        "w": lwidth,
        "o": opac,
        "col": color,
        "ft": rtype
    })

print(f"  Route line objects: {len(route_lines_js):,}")

# Trim heatmap if needed for file size
MAX_HEAT = 25000
if len(heatmap_pts) > MAX_HEAT:
    import random; random.seed(42)
    heatmap_pts = random.sample(heatmap_pts, MAX_HEAT)

# Pre-serialise to measure size
heat_json   = json.dumps(heatmap_pts)
routes_json = json.dumps(route_lines_js)
print(f"\nPayload sizes:")
print(f"  Heatmap : {len(heat_json)//1024:>4} KB  ({len(heatmap_pts):,} pts)")
print(f"  Routes  : {len(routes_json)//1024:>4} KB  ({len(route_lines_js):,} lines)")


In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 5 — Neighbourhood wealth data + transit scoring
# ═══════════════════════════════════════════════════════════════
from scipy import stats as sp_stats

WEALTH_RAW = [
    ("Kakkanad IT Hub",        10.0159, 76.3419, 0.87),
    ("Panampilly Nagar",        9.9737, 76.2930, 0.86),
    ("Marine Drive",            9.9793, 76.2808, 0.84),
    ("Edapally",               10.0269, 76.3088, 0.79),
    ("Vyttila",                 9.9647, 76.3140, 0.76),
    ("Palarivattom",           10.0023, 76.3069, 0.73),
    ("Kalmassery",             10.0469, 76.3227, 0.71),
    ("Kadavanthara",            9.9734, 76.2980, 0.74),
    ("Thrikkakara",            10.0200, 76.3300, 0.75),
    ("Ernakulam North",         9.9870, 76.2920, 0.63),
    ("Ernakulam South",         9.9662, 76.2820, 0.61),
    ("Thrippunithura",          9.9400, 76.3500, 0.59),
    ("Maradu",                  9.9490, 76.3280, 0.56),
    ("Aluva",                  10.1070, 76.3560, 0.54),
    ("Kalamassery Industrial", 10.0600, 76.3100, 0.48),
    ("Thevara",                 9.9600, 76.2920, 0.55),
    ("Tripunithura South",      9.9250, 76.3420, 0.51),
    ("Angamaly",               10.1960, 76.3870, 0.50),
    ("Fort Kochi",              9.9638, 76.2422, 0.38),
    ("Mattancherry",            9.9548, 76.2591, 0.34),
    ("Cheranalloor",           10.0040, 76.3670, 0.31),
    ("Thiruvankulam",           9.9900, 76.3800, 0.37),
    ("Pallipuram",             10.1390, 76.2170, 0.27),
    ("Vypeen Island North",    10.1450, 76.1960, 0.23),
    ("Njarakkal",              10.0760, 76.2350, 0.20),
    ("Mulavukad",              10.0960, 76.2190, 0.22),
    ("Cherai",                 10.1470, 76.2100, 0.25),
    ("Kumbalangi",              9.9300, 76.2780, 0.29),
]

NR = 0.018  # ~2 km radius
hm_lats = np.array([p[0] for p in heatmap_pts])
hm_lons = np.array([p[1] for p in heatmap_pts])
hm_vals = np.array([p[2] for p in heatmap_pts])

neighbourhoods = []
for name, lat, lon, wealth in WEALTH_RAW:
    dist = np.sqrt((hm_lats-lat)**2 + (hm_lons-lon)**2)
    mask = dist <= NR
    transit = float(np.percentile(hm_vals[mask], 75)) if mask.any() else 0.0
    transit = min(1.0, transit * 1.35)

    tt, wt = 0.35, 0.50
    if   transit >= tt and wealth >= wt: q = "WELL SERVED"
    elif transit >= tt:                  q = "TRANSIT DEPENDENT"
    elif wealth  >= wt:                  q = "CAR DEPENDENT"
    else:                                q = "DOUBLE DISADVANTAGE"

    neighbourhoods.append(dict(name=name, lat=lat, lon=lon,
                               wealth=round(wealth,3),
                               transit=round(transit,3),
                               quadrant=q))

t_v = [n["transit"] for n in neighbourhoods]
w_v = [n["wealth"]  for n in neighbourhoods]
r_val, p_val = sp_stats.pearsonr(t_v, w_v)

n_dd = sum(1 for n in neighbourhoods if n["quadrant"]=="DOUBLE DISADVANTAGE")
n_td = sum(1 for n in neighbourhoods if n["quadrant"]=="TRANSIT DEPENDENT")
n_ws = sum(1 for n in neighbourhoods if n["quadrant"]=="WELL SERVED")
n_cd = sum(1 for n in neighbourhoods if n["quadrant"]=="CAR DEPENDENT")

print(f"Pearson r = {r_val:+.4f}  p = {p_val:.4f}")
print(f"DD={n_dd}  TD={n_td}  CD={n_cd}  WS={n_ws}")
nbhd_json = json.dumps(neighbourhoods)


In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 6 — Generate HTML dashboard
#
#  FIXES vs previous version:
#  1. leaflet-heat (simpleheat + HeatLayer) embedded inline
#     — no CDN dependency, works 100% offline
#  2. Route colors are pure hex strings computed in Python
#     — no f-string survives into the JS template
#  3. Heatmap built from route shape geometry (not 610 stop pts)
#  4. Canvas renderer on the map for better performance
#  5. All JS template literals use ${} not Python format {}
# ═══════════════════════════════════════════════════════════════

# ── Inline JS libraries (no CDN needed) ──────────────────────
SIMPLEHEAT_JS = "(function(global){function simpleheat(canvas){if(!(this instanceof simpleheat))return new simpleheat(canvas);this._canvas=canvas=typeof canvas==='string'?document.getElementById(canvas):canvas;this._ctx=canvas.getContext('2d');this._width=canvas.width;this._height=canvas.height;this._max=1;this._data=[];}simpleheat.prototype={defaultRadius:25,defaultGradient:{0.4:'blue',0.6:'cyan',0.7:'lime',0.8:'yellow',1.0:'red'},data:function(data){this._data=data;return this;},max:function(max){this._max=max;return this;},add:function(point){this._data.push(point);return this;},clear:function(){this._data=[];return this;},radius:function(r,blur){blur=blur===undefined?15:blur;var circle=this._circle=this._createCanvas(),ctx=circle.getContext('2d'),r2=this._r=r+blur;circle.width=circle.height=r2*2;ctx.shadowOffsetX=ctx.shadowOffsetY=200;ctx.shadowBlur=blur;ctx.shadowColor='black';ctx.beginPath();ctx.arc(-200+r,0,r,0,Math.PI*2,true);ctx.closePath();ctx.fill();return this;},gradient:function(grad){var canvas=this._createCanvas(),ctx=canvas.getContext('2d'),gradient=ctx.createLinearGradient(0,0,0,256);canvas.width=1;canvas.height=256;for(var i in grad){gradient.addColorStop(+i,grad[i]);}ctx.fillStyle=gradient;ctx.fillRect(0,0,1,256);this._grad=ctx.getImageData(0,0,1,256).data;return this;},draw:function(minOpacity){if(!this._circle)this.radius(this.defaultRadius);if(!this._grad)this.gradient(this.defaultGradient);var ctx=this._ctx;ctx.clearRect(0,0,this._width,this._height);for(var i=0,len=this._data.length,p;i<len;i++){p=this._data[i];ctx.globalAlpha=Math.min(Math.max(p[2]/this._max,minOpacity===undefined?0.05:minOpacity),1);ctx.drawImage(this._circle,p[0]-this._r,p[1]-this._r);}var colored=ctx.getImageData(0,0,this._width,this._height);this._colorize(colored.data,this._grad);ctx.putImageData(colored,0,0);return this;},_colorize:function(pixels,gradient){for(var i=0,len=pixels.length,j;i<len;i+=4){j=pixels[i+3]*4;if(j){pixels[i]=gradient[j];pixels[i+1]=gradient[j+1];pixels[i+2]=gradient[j+2];}}},_createCanvas:function(){return document.createElement('canvas');}};if(typeof module!=='undefined')module.exports=simpleheat;else global.simpleheat=simpleheat;}(this));"

LEAFLET_HEAT_JS = """(function(){var HeatLayer=(L.Layer?L.Layer:L.Class).extend({initialize:function(latlngs,options){this._latlngs=latlngs;L.setOptions(this,options);},onAdd:function(map){this._map=map;if(!this._canvas)this._initCanvas();(this.options.pane?this.getPane():map._panes.overlayPane).appendChild(this._canvas);map.on('moveend',this._reset,this);if(map.options.zoomAnimation&&L.Browser.any3d)map.on('zoomanim',this._animateZoom,this);this._reset();},onRemove:function(map){(this.options.pane?this.getPane():map.getPanes().overlayPane).removeChild(this._canvas);map.off('moveend',this._reset,this);if(map.options.zoomAnimation)map.off('zoomanim',this._animateZoom,this);},addTo:function(map){map.addLayer(this);return this;},setLatLngs:function(latlngs){this._latlngs=latlngs;return this.redraw();},redraw:function(){if(this._heat&&!this._frame&&this._map&&!this._map._animating)this._frame=L.Util.requestAnimFrame(this._redraw,this);return this;},_initCanvas:function(){var canvas=this._canvas=L.DomUtil.create('canvas','leaflet-heatmap-layer leaflet-layer');var op=L.DomUtil.testProp(['transformOrigin','WebkitTransformOrigin','msTransformOrigin']);canvas.style[op]='50% 50%';var size=this._map.getSize();canvas.width=size.x;canvas.height=size.y;var animated=this._map.options.zoomAnimation&&L.Browser.any3d;L.DomUtil.addClass(canvas,'leaflet-zoom-'+(animated?'animated':'hide'));this._heat=simpleheat(canvas);this._heat.radius(this.options.radius||25,this.options.blur||15);if(this.options.gradient)this._heat.gradient(this.options.gradient);if(this.options.minOpacity)this._heat.minOpacity(this.options.minOpacity);},_reset:function(){var tl=this._map.containerPointToLayerPoint([0,0]);L.DomUtil.setPosition(this._canvas,tl);var size=this._map.getSize();if(this._heat._width!==size.x){this._canvas.width=this._heat._width=size.x;}if(this._heat._height!==size.y){this._canvas.height=this._heat._height=size.y;}this._redraw();},_redraw:function(){this._frame=null;if(!this._heat)return;var d=[],r=this._heat._r,size=this._map.getSize(),bounds=new L.Bounds(L.point([-r,-r]),size.add([r,r])),max=this.options.max||1,maxZoom=this.options.maxZoom||this._map.getMaxZoom(),v=1/Math.pow(2,Math.max(0,Math.min(maxZoom-this._map.getZoom(),12))),cellSize=r/2,grid=[],panePos=this._map._getMapPanePos(),ox=panePos.x%cellSize,oy=panePos.y%cellSize;for(var i=0,len=this._latlngs.length;i<len;i++){var ll=this._latlngs[i];var p=this._map.latLngToContainerPoint(ll);if(p.x>bounds.max.x||p.x<bounds.min.x||p.y>bounds.max.y||p.y<bounds.min.y)continue;var x=Math.floor((p.x-ox)/cellSize)+2,y=Math.floor((p.y-oy)/cellSize)+2;var alt=Array.isArray(ll)?+(ll[2]||1):1;var k=alt*v;grid[y]=grid[y]||[];var cell=grid[y][x];if(!cell){grid[y][x]=[p.x,p.y,k];}else{cell[0]=(cell[0]*cell[2]+p.x*k)/(cell[2]+k);cell[1]=(cell[1]*cell[2]+p.y*k)/(cell[2]+k);cell[2]+=k;}}for(var i=0;i<grid.length;i++){if(grid[i])for(var j=0;j<grid[i].length;j++){var cell=grid[i][j];if(cell)d.push([cell[0],cell[1],Math.min(cell[2],max)]);}}this._heat.data(d).draw(this.options.minOpacity);},_animateZoom:function(e){var scale=this._map.getZoomScale(e.zoom),offset=this._map._getCenterOffset(e.center)._multiplyBy(-scale).subtract(this._map._getMapPanePos());if(L.DomUtil.setTransform)L.DomUtil.setTransform(this._canvas,offset,scale);else this._canvas.style[L.DomUtil.TRANSFORM]=L.DomUtil.getTranslateString(offset)+' scale('+scale+')';}});L.heatLayer=function(latlngs,options){return new HeatLayer(latlngs,options);};})();"""

# ── Build the HTML ────────────────────────────────────────────
r_str   = f"{r_val:+.3f}"
n_routes_str = str(len(shape_lines))
n_stops_str  = str(len(stops_df))
n_heat_str   = str(len(heatmap_pts))

HTML = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Kochi Urban Equity Dashboard</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Syne:wght@400;600;700;800&family=DM+Sans:wght@300;400;500&display=swap" rel="stylesheet">
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.css"/>
<script src="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.js"></script>
<script>
/* simpleheat — embedded inline, no CDN */
""" + SIMPLEHEAT_JS + """
/* Leaflet.heat — embedded inline, no CDN */
""" + LEAFLET_HEAT_JS + """
</script>
<style>
:root{
  --bg:#070c13;--surf:#0e1520;--surf2:#14203080;--surf3:#1a2840;
  --bdr:#1e2e45;--text:#d8e4f2;--muted:#4e6080;--dim:#2a3a55;
  --lime:#c2f000;--green:#00df7a;--blue:#3d8fff;--cyan:#00ccff;
  --amber:#ffaa00;--red:#ff3650;--sidebar:340px;
}
*{box-sizing:border-box;margin:0;padding:0}
html,body{height:100%;font-family:'DM Sans',sans-serif;background:var(--bg);color:var(--text);overflow:hidden}
#app{display:flex;flex-direction:column;height:100vh}

/* TOP BAR */
#topbar{
  height:50px;display:flex;align-items:center;gap:12px;padding:0 16px;
  background:var(--surf);border-bottom:1px solid var(--bdr);flex-shrink:0;z-index:1000;
}
.logo{font-family:'Syne',sans-serif;font-weight:800;font-size:14px;letter-spacing:-.3px;display:flex;align-items:center;gap:6px}
.logo .a{color:var(--lime)}.logo .b{color:var(--muted);font-weight:400}
.badge{font-size:9px;font-weight:600;letter-spacing:1.8px;text-transform:uppercase;padding:2px 8px;border:1px solid var(--bdr);border-radius:20px;color:var(--muted)}
.sp{flex:1}
.src{font-size:9px;color:var(--dim)}

/* LAYER TABS */
#ltabs{display:flex;gap:2px;background:var(--bg);border-radius:7px;padding:3px}
.lt{font-size:10.5px;font-weight:500;padding:4px 12px;border-radius:5px;border:none;cursor:pointer;background:transparent;color:var(--muted);transition:all .15s}
.lt.on{background:var(--lime);color:#070c13;font-weight:700}

/* MAIN */
#main{display:flex;flex:1;overflow:hidden;position:relative}
#map-wrap{flex:1;position:relative}
#map{width:100%;height:100%}

/* SIDEBAR */
#sb{
  width:var(--sidebar);flex-shrink:0;
  background:var(--surf);border-left:1px solid var(--bdr);
  display:flex;flex-direction:column;overflow:hidden;
}
#sb-hd{padding:13px 15px 10px;border-bottom:1px solid var(--bdr);flex-shrink:0}
#sb-hd h2{font-family:'Syne',sans-serif;font-size:10px;font-weight:700;letter-spacing:2px;text-transform:uppercase;color:var(--muted);margin-bottom:9px}
#qf{display:flex;flex-wrap:wrap;gap:4px}
.qp{font-size:9px;font-weight:600;letter-spacing:.5px;text-transform:uppercase;padding:3px 9px;border-radius:20px;cursor:pointer;border:1.5px solid transparent;transition:all .14s}
.qp[data-q="ALL"]{border-color:var(--muted);color:var(--muted)}
.qp[data-q="WELL SERVED"]{border-color:var(--green);color:var(--green)}
.qp[data-q="TRANSIT DEPENDENT"]{border-color:var(--blue);color:var(--blue)}
.qp[data-q="CAR DEPENDENT"]{border-color:var(--amber);color:var(--amber)}
.qp[data-q="DOUBLE DISADVANTAGE"]{border-color:var(--red);color:var(--red)}
.qp.on[data-q="ALL"]{background:var(--muted);color:#070c13}
.qp.on[data-q="WELL SERVED"]{background:var(--green);color:#070c13}
.qp.on[data-q="TRANSIT DEPENDENT"]{background:var(--blue);color:#fff}
.qp.on[data-q="CAR DEPENDENT"]{background:var(--amber);color:#070c13}
.qp.on[data-q="DOUBLE DISADVANTAGE"]{background:var(--red);color:#fff}

/* KPI ROW */
#kpi{display:grid;grid-template-columns:1fr 1fr;gap:1px;background:var(--bdr);border-bottom:1px solid var(--bdr)}
.kk{background:var(--surf);padding:10px 13px}
.kv{font-family:'Syne',sans-serif;font-size:23px;font-weight:800;letter-spacing:-1px;line-height:1}
.kl{font-size:8px;text-transform:uppercase;letter-spacing:1px;color:var(--muted);margin-top:2px}
.kk.red .kv{color:var(--red)}.kk.blue .kv{color:var(--blue)}
.kk.amber .kv{color:var(--amber)}.kk.green .kv{color:var(--green)}

/* SCROLL BODY */
#sb-body{flex:1;overflow-y:auto}
#sb-body::-webkit-scrollbar{width:3px}
#sb-body::-webkit-scrollbar-thumb{background:var(--bdr);border-radius:2px}

/* ALERT */
#alrt{margin:9px 11px 3px;padding:9px 11px;border-radius:7px;
  background:rgba(255,54,80,.08);border:1px solid rgba(255,54,80,.28);
  display:flex;gap:7px;align-items:flex-start}
#alrt .ai{font-size:12px;flex-shrink:0;line-height:1.5}
#alrt .at{font-size:10.5px;line-height:1.55;color:#ffaabb}
#alrt .at strong{color:var(--red)}

/* NEIGHBOURHOOD LIST */
#nl{padding:7px 9px}
.nc{padding:9px 11px;border-radius:7px;margin-bottom:5px;border:1px solid var(--bdr);cursor:pointer;transition:all .14s;background:rgba(14,21,32,.6);display:flex;align-items:center;gap:8px}
.nc:hover,.nc.sel{border-color:var(--lime);background:var(--surf3)}
.ncdot{width:9px;height:9px;border-radius:50%;flex-shrink:0}
.nci{flex:1;min-width:0}
.ncn{font-size:11px;font-weight:500;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.ncq{font-size:8px;color:var(--muted);text-transform:uppercase;letter-spacing:.5px;margin-top:1px}
.ncb{width:60px;flex-shrink:0;display:flex;flex-direction:column;gap:3px}
.br{display:flex;align-items:center;gap:3px}
.bl{font-size:6.5px;color:var(--dim);width:6px;text-transform:uppercase}
.bt{flex:1;height:3px;background:var(--bdr);border-radius:2px;overflow:hidden}
.bf{height:100%;border-radius:2px;transition:width .4s}
.bft{background:var(--blue)}.bfw{background:var(--amber)}

/* DETAIL PANEL */
#dp{display:none;position:absolute;bottom:0;left:0;right:0;background:var(--surf);border-top:2px solid var(--lime);z-index:800;animation:su .2s ease}
@keyframes su{from{transform:translateY(14px);opacity:0}to{transform:none;opacity:1}}
#dp.on{display:block}
#dp-hd{padding:13px 15px 9px;border-bottom:1px solid var(--bdr);display:flex;justify-content:space-between;align-items:flex-start}
#dpn{font-family:'Syne',sans-serif;font-size:16px;font-weight:800;letter-spacing:-.4px;line-height:1.1}
#dpq{font-size:8.5px;font-weight:700;letter-spacing:1.5px;text-transform:uppercase;margin-top:3px}
#dpx{cursor:pointer;color:var(--muted);font-size:15px;padding:2px 4px}
#dpx:hover{color:var(--text)}
#dp-bd{padding:11px 15px;display:grid;grid-template-columns:1fr 1fr;gap:9px}
.dm{background:rgba(14,21,32,.7);border-radius:7px;padding:10px 12px}
.dm-l{font-size:8px;text-transform:uppercase;letter-spacing:1px;color:var(--muted);margin-bottom:3px}
.dm-v{font-family:'Syne',sans-serif;font-size:24px;font-weight:800;letter-spacing:-1px;line-height:1}
.dm-bt{height:4px;background:var(--bdr);border-radius:2px;margin-top:5px;overflow:hidden}
.dm-bf{height:100%;border-radius:2px;transition:width .5s .1s}
#dpvrd{grid-column:1/-1;padding:10px 12px;border-radius:7px;border-left:3px solid}
#dpvrd .vt{font-family:'Syne',sans-serif;font-size:10px;font-weight:700;text-transform:uppercase;letter-spacing:1px;margin-bottom:4px}
#dpvrd .vb{font-size:11px;line-height:1.6;color:#b0c0d8}
#dpact{grid-column:1/-1;background:rgba(14,21,32,.7);border-radius:7px;padding:10px 12px}
#dpact .at2{font-size:8px;text-transform:uppercase;letter-spacing:1px;color:var(--lime);font-weight:700;margin-bottom:4px}
#dpact .ab{font-size:11px;line-height:1.6;color:#b0c0d8}

/* MAP LEGENDS */
.mleg{position:absolute;z-index:700;background:rgba(7,12,19,.88);backdrop-filter:blur(12px);border:1px solid var(--bdr);border-radius:9px;padding:11px 13px}
#leg-eq{bottom:16px;left:16px}
#leg-eq h4{font-family:'Syne',sans-serif;font-size:8px;font-weight:700;text-transform:uppercase;letter-spacing:1.5px;color:var(--muted);margin-bottom:7px}
.lr{display:flex;align-items:center;gap:7px;margin-bottom:5px}
.ld{width:10px;height:10px;border-radius:50%;flex-shrink:0}
.ll{font-size:9px;line-height:1.3}
.ll strong{display:block;font-weight:600}
.ll small{color:var(--muted);font-size:8px}
#leg-ht{bottom:16px;right:calc(var(--sidebar)+16px);display:none;min-width:155px}
#leg-ht h4{font-family:'Syne',sans-serif;font-size:8px;font-weight:700;text-transform:uppercase;letter-spacing:1.5px;color:var(--muted);margin-bottom:7px}
.hg{width:129px;height:7px;border-radius:4px;background:linear-gradient(to right,#0a0000,#3d0000,#8b0000,#ff4400,#ffcc00,#88ff44,#00ccff);margin-bottom:5px}
.hls{display:flex;justify-content:space-between}
.hls span{font-size:7.5px;color:var(--muted)}
#leg-rt{bottom:16px;right:calc(var(--sidebar)+16px);display:none}
#leg-rt h4{font-family:'Syne',sans-serif;font-size:8px;font-weight:700;text-transform:uppercase;letter-spacing:1.5px;color:var(--muted);margin-bottom:7px}

/* MINI SCATTER */
#scw{padding:9px 11px 10px;border-top:1px solid var(--bdr)}
#scw h3{font-family:'Syne',sans-serif;font-size:8px;font-weight:700;text-transform:uppercase;letter-spacing:1.5px;color:var(--muted);margin-bottom:7px}
#sc{width:100%;display:block}

.leaflet-container{background:#070c13!important}
</style>
</head>
<body>
<div id="app">

<div id="topbar">
  <div class="logo"><span class="a">&#9632;</span>&nbsp;KOCHI <span class="b">Urban Equity</span></div>
  <div class="badge">Policy Tool</div>
  <div class="sp"></div>
  <div id="ltabs">
    <button class="lt on" id="lt-eq"  onclick="setLayer('equity')">Equity Map</button>
    <button class="lt"    id="lt-ht"  onclick="setLayer('heat')">Transit Density</button>
    <button class="lt"    id="lt-rt"  onclick="setLayer('routes')">Route Network</button>
    <button class="lt"    id="lt-all" onclick="setLayer('all')">All Layers</button>
  </div>
  <div class="src">GTFS mdb-1835 &nbsp;|&nbsp; """ + n_routes_str + """ routes &nbsp;|&nbsp; """ + n_stops_str + """ stops &nbsp;|&nbsp; Census 2011</div>
</div>

<div id="main">
  <div id="map-wrap">
    <div id="map"></div>
    <div class="mleg" id="leg-eq">
      <h4>Neighbourhood Status</h4>
      <div class="lr"><div class="ld" style="background:#00df7a"></div><div class="ll"><strong>Well Served</strong><small>Good transit + higher wealth</small></div></div>
      <div class="lr"><div class="ld" style="background:#3d8fff"></div><div class="ll"><strong>Transit Dependent</strong><small>Low wealth, relies on buses</small></div></div>
      <div class="lr"><div class="ld" style="background:#ffaa00"></div><div class="ll"><strong>Car Dependent</strong><small>Wealth present, transit absent</small></div></div>
      <div class="lr"><div class="ld" style="background:#ff3650"></div><div class="ll"><strong>Double Disadvantage</strong><small>Low wealth AND low transit</small></div></div>
    </div>
    <div class="mleg" id="leg-ht">
      <h4>Transit Density (""" + n_heat_str + """ pts)</h4>
      <div class="hg"></div>
      <div class="hls"><span>Sparse</span><span>Dense</span></div>
      <div style="font-size:7.5px;color:var(--muted);margin-top:4px">Route geometry weighted by service frequency</div>
    </div>
    <div class="mleg" id="leg-rt">
      <h4>Route Network</h4>
      <div class="lr"><div style="width:20px;height:3px;background:#00ccff;border-radius:2px;flex-shrink:0"></div><div class="ll" style="font-size:8.5px">Ferry routes</div></div>
      <div class="lr"><div style="width:20px;height:2px;background:#3060a0;border-radius:1px;flex-shrink:0"></div><div class="ll" style="font-size:8.5px">Low freq. bus</div></div>
      <div class="lr"><div style="width:20px;height:3px;background:#3d8fff;border-radius:1px;flex-shrink:0"></div><div class="ll" style="font-size:8.5px">High freq. bus</div></div>
      <div style="font-size:7.5px;color:var(--muted);margin-top:4px">Width &amp; brightness = service frequency</div>
    </div>
  </div>

  <div id="sb">
    <div id="sb-hd">
      <h2>Filter by Status</h2>
      <div id="qf">
        <div class="qp on" data-q="ALL"                onclick="filterQ(this)">All</div>
        <div class="qp"    data-q="WELL SERVED"        onclick="filterQ(this)">Well Served</div>
        <div class="qp"    data-q="TRANSIT DEPENDENT"  onclick="filterQ(this)">Transit Dep.</div>
        <div class="qp"    data-q="CAR DEPENDENT"      onclick="filterQ(this)">Car Dep.</div>
        <div class="qp"    data-q="DOUBLE DISADVANTAGE" onclick="filterQ(this)">&#9888; Double Disadv.</div>
      </div>
    </div>
    <div id="sb-body">
      <div id="kpi">
        <div class="kk red">  <div class="kv" id="k-dd">""" + str(n_dd) + """</div><div class="kl">Double Disadvantage</div></div>
        <div class="kk blue"> <div class="kv" id="k-td">""" + str(n_td) + """</div><div class="kl">Transit Dependent</div></div>
        <div class="kk amber"><div class="kv" id="k-cd">""" + str(n_cd) + """</div><div class="kl">Car Dependent</div></div>
        <div class="kk green"><div class="kv" id="k-ws">""" + str(n_ws) + """</div><div class="kl">Well Served</div></div>
      </div>
      <div id="alrt">
        <div class="ai">&#9888;</div>
        <div class="at"><strong>Njarakkal</strong> — lowest wealth (0.20) and near-zero transit access in the dataset. Island geography makes this Kochi's most urgent equity gap.</div>
      </div>
      <div id="nl"></div>
      <div id="scw">
        <h3>Transit vs Wealth &nbsp;|&nbsp; 28 neighbourhoods &nbsp;|&nbsp; r&nbsp;=&nbsp;""" + r_str + """</h3>
        <canvas id="sc" width="308" height="170"></canvas>
      </div>
    </div>
    <div id="dp">
      <div id="dp-hd">
        <div><div id="dpn"></div><div id="dpq"></div></div>
        <div id="dpx" onclick="closeDP()">&#10005;</div>
      </div>
      <div id="dp-bd">
        <div class="dm"><div class="dm-l">Transit Access</div><div class="dm-v" id="dpvt"></div><div class="dm-bt"><div class="dm-bf" id="dpbt" style="background:var(--blue)"></div></div></div>
        <div class="dm"><div class="dm-l">Wealth Index</div> <div class="dm-v" id="dpvw"></div><div class="dm-bt"><div class="dm-bf" id="dpbw" style="background:var(--amber)"></div></div></div>
        <div id="dpvrd"><div class="vt">What This Means</div><div class="vb" id="dpt"></div></div>
        <div id="dpact"><div class="at2">&#8594; Policy Action</div><div class="ab" id="dpa"></div></div>
      </div>
    </div>
  </div>
</div>
</div>

<script>
/* ── DATA (generated by Python, all values are pure literals) ── */
const NBHD   = """ + nbhd_json   + """;
const HEAT   = """ + heat_json   + """;
const ROUTES = """ + routes_json + """;

const QC = {
  "WELL SERVED":         "#00df7a",
  "TRANSIT DEPENDENT":   "#3d8fff",
  "CAR DEPENDENT":       "#ffaa00",
  "DOUBLE DISADVANTAGE": "#ff3650"
};
const VERDICT = {
  "WELL SERVED":         "This neighbourhood has strong transit coverage AND relatively high wealth. Residents have both transport options and economic resources. The network is working well here.",
  "TRANSIT DEPENDENT":   "Lower-wealth residents rely on public buses as their primary — often only — way to reach jobs, schools, and hospitals. Any service cuts here have disproportionate human consequences.",
  "CAR DEPENDENT":       "Despite higher wealth, this area has weak transit coverage. Residents absorb the gap with private cars. This is a missed ridership opportunity and a vulnerability if fuel costs rise.",
  "DOUBLE DISADVANTAGE": "CRITICAL: Low wealth AND low transit access. Residents cannot afford cars and buses don't adequately reach them. Transit investment here delivers the highest equity return per rupee spent."
};
const ACTIONS = {
  "WELL SERVED":         "Maintain current frequency. Consider extending express corridors to link this hub with underserved peripheral zones. Monitor for displacement of transit-dependent residents.",
  "TRANSIT DEPENDENT":   "Protect existing routes — do not reduce frequency. Audit last-mile gaps. Consider subsidised feeder rickshaw integration. Engage communities on unmet trip needs.",
  "CAR DEPENDENT":       "Pilot a new bus route or extend an existing corridor into this zone. Conduct a demand survey before committing capital. Parking-linked transit incentives may help.",
  "DOUBLE DISADVANTAGE": "PRIORITY INTERVENTION. Introduce at minimum one reliable daily route with guaranteed frequency. Explore ferry connectivity if near waterfront. Fast-track in next budget cycle."
};

/* ── MAP ── */
const map = L.map('map', {center:[10.02,76.30], zoom:11, preferCanvas:true});
L.tileLayer('https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png',{
  attribution:'&copy; OpenStreetMap &copy; CARTO', subdomains:'abcd', maxZoom:20
}).addTo(map);

/* ── LAYERS ── */
const lgEq = L.layerGroup();
const lgRt = L.layerGroup();
const lgHt = L.heatLayer(HEAT, {
  radius: 16,
  blur:   20,
  minOpacity: 0.35,
  maxZoom: 16,
  gradient: {
    0.0: '#000000',
    0.1: '#0a0015',
    0.25:'#3d0000',
    0.4: '#8b0000',
    0.55:'#ff4400',
    0.70:'#ffcc00',
    0.85:'#88ff44',
    1.0: '#00ccff'
  }
});

/* Build route polylines */
ROUTES.forEach(function(r) {
  L.polyline(r.c, {
    color:   r.col,
    weight:  r.w,
    opacity: r.o,
    smoothFactor: 1.5,
    interactive: false
  }).addTo(lgRt);
});

/* Build equity markers */
const mkrs = {};
NBHD.forEach(function(n, i) {
  var color = QC[n.quadrant];
  var radius = 9 + n.wealth * 13;
  var m = L.circleMarker([n.lat, n.lon], {
    radius: radius, color: color, fillColor: color,
    fillOpacity: 0.85, weight: 2, opacity: 0.9
  });
  m.bindTooltip(
    '<div style="font-family:DM Sans,sans-serif;min-width:165px;padding:2px">'
    + '<div style="font-weight:700;font-size:13px;margin-bottom:3px">' + n.name + '</div>'
    + '<div style="font-size:8.5px;text-transform:uppercase;letter-spacing:1px;color:#7a8aaa;margin-bottom:6px">' + n.quadrant + '</div>'
    + '<div style="display:grid;grid-template-columns:1fr 1fr;gap:4px">'
    + '<div style="background:#0e1520;padding:5px 7px;border-radius:5px">'
    + '<div style="font-size:7.5px;color:#4e6080;margin-bottom:1px">TRANSIT</div>'
    + '<div style="font-size:17px;font-weight:800;color:#3d8fff;letter-spacing:-0.5px">' + Math.round(n.transit*100) + '%</div>'
    + '</div>'
    + '<div style="background:#0e1520;padding:5px 7px;border-radius:5px">'
    + '<div style="font-size:7.5px;color:#4e6080;margin-bottom:1px">WEALTH</div>'
    + '<div style="font-size:17px;font-weight:800;color:#ffaa00;letter-spacing:-0.5px">' + Math.round(n.wealth*100) + '%</div>'
    + '</div></div></div>',
    {sticky:false, className:'dkt', offset:[0,-8]}
  );
  m.on('click', function(){ selectN(i); });
  lgEq.addLayer(m);
  mkrs[i] = m;
});
lgEq.addTo(map);

/* ── LAYER SWITCHING ── */
var curLayer = 'equity';
function setLayer(mode) {
  curLayer = mode;
  ['lt-eq','lt-ht','lt-rt','lt-all'].forEach(function(id){
    document.getElementById(id).classList.remove('on');
  });
  var idmap = {equity:'lt-eq', heat:'lt-ht', routes:'lt-rt', all:'lt-all'};
  document.getElementById(idmap[mode]).classList.add('on');

  [lgEq, lgHt, lgRt].forEach(function(l){ try{ map.removeLayer(l); }catch(e){} });
  document.getElementById('leg-eq').style.display = 'none';
  document.getElementById('leg-ht').style.display = 'none';
  document.getElementById('leg-rt').style.display = 'none';

  if (mode === 'equity')  { lgEq.addTo(map); document.getElementById('leg-eq').style.display='block'; }
  if (mode === 'heat')    { lgHt.addTo(map); document.getElementById('leg-ht').style.display='block'; }
  if (mode === 'routes')  { lgRt.addTo(map); document.getElementById('leg-rt').style.display='block'; }
  if (mode === 'all')     {
    lgRt.addTo(map); lgHt.addTo(map); lgEq.addTo(map);
    document.getElementById('leg-eq').style.display='block';
    document.getElementById('leg-ht').style.display='block';
  }
}
document.getElementById('leg-eq').style.display='block';

/* ── SIDEBAR LIST ── */
var activeQ = 'ALL', selIdx = null;
function buildList(q) {
  var el = document.getElementById('nl'); el.innerHTML = '';
  NBHD.forEach(function(n, i) {
    if (q !== 'ALL' && n.quadrant !== q) return;
    var c = QC[n.quadrant];
    var d = document.createElement('div');
    d.className = 'nc' + (selIdx === i ? ' sel' : '');
    d.innerHTML =
      '<div class="ncdot" style="background:' + c + '"></div>'
      + '<div class="nci"><div class="ncn">' + n.name + '</div><div class="ncq">' + n.quadrant + '</div></div>'
      + '<div class="ncb">'
      + '<div class="br"><span class="bl">T</span><div class="bt"><div class="bf bft" style="width:' + (n.transit*100) + '%"></div></div></div>'
      + '<div class="br"><span class="bl">W</span><div class="bt"><div class="bf bfw" style="width:' + (n.wealth*100)  + '%"></div></div></div>'
      + '</div>';
    d.onclick = (function(idx){ return function(){ selectN(idx); }; })(i);
    el.appendChild(d);
  });
}
function filterQ(el) {
  document.querySelectorAll('.qp').forEach(function(e){ e.classList.remove('on'); });
  el.classList.add('on'); activeQ = el.dataset.q;
  buildList(activeQ);
  NBHD.forEach(function(n, i) {
    var vis = (activeQ === 'ALL' || n.quadrant === activeQ);
    mkrs[i].setStyle({fillOpacity: vis ? 0.85 : 0.06, opacity: vis ? 0.9 : 0.1});
  });
}

/* ── DETAIL PANEL ── */
function selectN(i) {
  selIdx = i; var n = NBHD[i]; var c = QC[n.quadrant];
  NBHD.forEach(function(_,j){ mkrs[j].setStyle({weight: j===i ? 3.5 : 2}); });
  map.flyTo([n.lat, n.lon], 13.5, {duration: 0.65});
  document.getElementById('dpn').textContent = n.name;
  document.getElementById('dpq').textContent = n.quadrant;
  document.getElementById('dpq').style.color = c;
  document.getElementById('dpvt').textContent = Math.round(n.transit*100) + '%';
  document.getElementById('dpvw').textContent = Math.round(n.wealth*100)  + '%';
  document.getElementById('dpbt').style.width = (n.transit*100) + '%';
  document.getElementById('dpbw').style.width = (n.wealth*100)  + '%';
  document.getElementById('dpt').textContent = VERDICT[n.quadrant];
  document.getElementById('dpa').textContent = ACTIONS[n.quadrant];
  var v = document.getElementById('dpvrd');
  v.style.background = c + '18'; v.style.borderColor = c;
  document.getElementById('dp').classList.add('on');
  buildList(activeQ);
}
function closeDP() {
  document.getElementById('dp').classList.remove('on');
  selIdx = null; buildList(activeQ);
  map.flyTo([10.02,76.30], 11, {duration:0.65});
}

/* ── MINI SCATTER ── */
function drawScatter() {
  var cv = document.getElementById('sc');
  var ctx = cv.getContext('2d');
  var W = cv.width, H = cv.height, P = 24;
  ctx.fillStyle = '#0e1520'; ctx.fillRect(0,0,W,H);
  ctx.strokeStyle = '#1a2840'; ctx.lineWidth = 0.5;
  for (var v = 0; v <= 1; v += 0.25) {
    var x = P + v*(W-P*2), y = H-P - v*(H-P*2);
    ctx.beginPath(); ctx.moveTo(x,P);   ctx.lineTo(x,H-P); ctx.stroke();
    ctx.beginPath(); ctx.moveTo(P,y);   ctx.lineTo(W-P,y); ctx.stroke();
  }
  ctx.strokeStyle='#2a3a58'; ctx.lineWidth=1; ctx.setLineDash([3,3]);
  var qx=P+0.35*(W-P*2), qy=H-P-0.5*(H-P*2);
  ctx.beginPath(); ctx.moveTo(qx,P);   ctx.lineTo(qx,H-P); ctx.stroke();
  ctx.beginPath(); ctx.moveTo(P,qy);   ctx.lineTo(W-P,qy); ctx.stroke();
  ctx.setLineDash([]);
  ctx.fillStyle='#2a3a55'; ctx.font='7px DM Sans'; ctx.textAlign='center';
  ctx.fillText('Transit \u2192',W/2,H-3);
  ctx.save(); ctx.translate(9,H/2); ctx.rotate(-Math.PI/2);
  ctx.fillText('Wealth \u2192',0,0); ctx.restore();
  NBHD.forEach(function(n) {
    var px=P+n.transit*(W-P*2), py=H-P-n.wealth*(H-P*2);
    ctx.beginPath(); ctx.arc(px,py,4.5,0,Math.PI*2);
    ctx.fillStyle=QC[n.quadrant]; ctx.globalAlpha=0.88; ctx.fill(); ctx.globalAlpha=1;
  });
}

/* Tooltip CSS */
var ts = document.createElement('style');
ts.textContent = '.dkt{background:#0e1520;border:1px solid #1e2e45;border-radius:8px;padding:9px 11px;color:#d8e4f2;box-shadow:0 8px 28px rgba(0,0,0,.65)}.dkt::before{display:none}';
document.head.appendChild(ts);

buildList('ALL');
drawScatter();
</script>
</body>
</html>"""

out_path = os.path.join(OUT_DIR, "kochi_equity_dashboard_v3.html")
with open(out_path, "w", encoding="utf-8") as f:
    f.write(HTML)

sz = os.path.getsize(out_path) // 1024
print(f"HTML written: {out_path}")
print(f"File size   : {sz} KB")
print(f"\nWhat's fixed vs previous version:")
print(f"  [OK] leaflet-heat embedded inline — no CDN, always works")
print(f"  [OK] Route colors computed in Python as pure hex strings")
print(f"  [OK] Heatmap uses {len(heatmap_pts):,} route geometry points (not 610 stops)")
print(f"  [OK] All JS uses string concatenation, no Python f-strings leaking into JS")


In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 7 — Download the HTML file
# ═══════════════════════════════════════════════════════════════
try:
    from google.colab import files
    files.download(out_path)
    print("Download started: kochi_equity_dashboard_v3.html")
    print("Open it in Chrome or Firefox — no internet required.")
    print("\nIf the download doesn't start, run this cell again.")
except ImportError:
    print(f"Not in Colab. File is at: {out_path}")
